In [ ]:
!pip install transformers accelerate bitsandbytes -q

In [ ]:
from google.colab import files
uploaded=files.upload()

Saving dataset_test.json to dataset_test.json


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizer loaded!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded!


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded!")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!


In [ ]:
import json

with open("dataset_test.json", "r") as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} test examples")
print(f"First example keys: {list(test_data[0].keys())}")

Loaded 80 test examples
First example keys: ['input', 'output']


In [ ]:
print(test_data)

[{'input': 'Nothing Ear (2) transparent design earbuds. 11.6mm custom driver, LHDC 5.0, personalized ANC, IP54 rated. 4 hours + 22.5 with case. $149. New.', 'output': {'product_name': 'Nothing Ear (2)', 'brand': 'Nothing', 'category': 'Earbuds', 'price': {'amount': 149.0, 'currency': 'USD'}, 'key_specs': {'type': 'true wireless earbuds', 'driver_size': '11.6mm', 'battery_life': '4 hours (22.5 with case)', 'features': 'ANC, LHDC 5.0, IP54'}, 'condition': 'new'}}, {'input': 'Nikon COOLPIX P1000 superzoom. 16MP BSI CMOS, 24-3000mm equivalent 125x optical zoom, 4K UHD video, RAW support. ₹74,950. New.', 'output': {'product_name': 'Nikon COOLPIX P1000', 'brand': 'Nikon', 'category': 'Camera', 'price': {'amount': 74950.0, 'currency': 'INR'}, 'key_specs': {'type': 'bridge/superzoom', 'sensor': '16MP BSI CMOS', 'zoom': '125x optical (24-3000mm equiv)', 'video': '4K UHD', 'features': 'RAW support'}, 'condition': 'new'}}, {'input': 'Unleash your gaming potential with the AOC AGON AG274QZM! 27" Q

In [ ]:
system_prompt = """You are a product data extraction assistant. Extract structured information from the given product description and return a valid JSON object with these fields:
- product_name: string
- brand: string or null if unknown
- category: string
- price: {"amount": number, "currency": string} or null if not mentioned
- key_specs: object with relevant specifications
- condition: "new", "used", or "refurbished"

Return ONLY the JSON object, no other text."""

def build_prompt(product_description):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Extract product data from this description:\n\n{product_description}"}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    return prompt

# Test it on the first example
sample_prompt = build_prompt(test_data[0]["input"])
print(sample_prompt)

<|im_start|>system
You are a product data extraction assistant. Extract structured information from the given product description and return a valid JSON object with these fields:
- product_name: string
- brand: string or null if unknown
- category: string
- price: {"amount": number, "currency": string} or null if not mentioned
- key_specs: object with relevant specifications
- condition: "new", "used", or "refurbished"

Return ONLY the JSON object, no other text.<|im_end|>
<|im_start|>user
Extract product data from this description:

Nothing Ear (2) transparent design earbuds. 11.6mm custom driver, LHDC 5.0, personalized ANC, IP54 rated. 4 hours + 22.5 with case. $149. New.<|im_end|>
<|im_start|>assistant



In [ ]:
def generate_response(prompt):
   # print(prompt)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
   # print(inputs)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
        )

    # Decode only the NEW tokens (skip the prompt tokens)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response

# Test on first example
test_response = generate_response(sample_prompt)
print(test_response)


<|im_start|>system
You are a product data extraction assistant. Extract structured information from the given product description and return a valid JSON object with these fields:
- product_name: string
- brand: string or null if unknown
- category: string
- price: {"amount": number, "currency": string} or null if not mentioned
- key_specs: object with relevant specifications
- condition: "new", "used", or "refurbished"

Return ONLY the JSON object, no other text.<|im_end|>
<|im_start|>user
Extract product data from this description:

Nothing Ear (2) transparent design earbuds. 11.6mm custom driver, LHDC 5.0, personalized ANC, IP54 rated. 4 hours + 22.5 with case. $149. New.<|im_end|>
<|im_start|>assistant

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,   1985,    821,  32189,
          17847,     13,  22826,  32930,   1995,    504,    279,   2661,   1985,
           4008,    323,    470,    264,   2697,   4718,   1633,    448,   1493,
           5043,    510,  

TypeError: print() got an unexpected keyword argument 'input_ids'

In [ ]:
results = []

for i, example in enumerate(test_data):
    prompt = build_prompt(example["input"])
    response = generate_response(prompt)

    results.append({
        "input": example["input"],
        "expected": example["output"],
        "predicted": response
    })

    if (i + 1) % 10 == 0:
        print(f"Completed {i + 1}/80")

print("All 80 examples done!")


Completed 10/80
Completed 20/80


KeyboardInterrupt: 

In [ ]:
import re

def parse_json_response(response_text):
    # Step 1: Try parsing the response directly
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass

    # Step 2: Try extracting JSON from markdown code block
    code_block = re.search(r"```(?:json)?\s*(.*?)\s*```", response_text, re.DOTALL)
    if code_block:
        try:
            return json.loads(code_block.group(1))
        except json.JSONDecodeError:
            pass

    # Step 3: Try finding anything between { and }
    brace_match = re.search(r"\{.*\}", response_text, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group())
        except json.JSONDecodeError:
            pass

    # Nothing worked
    return None

# Test it
sample_parsed = parse_json_response(results[0]["predicted"])
print(type(sample_parsed))
print(json.dumps(sample_parsed, indent=2))


In [ ]:
def normalize(value):
    if value is None:
        return None
    if isinstance(value, str):
        return value.strip().lower()
    return value

def compare_price(predicted, expected):
    if predicted is None and expected is None:
        return True
    if predicted is None or expected is None:
        return False
    if not isinstance(predicted, dict) or not isinstance(expected, dict):
        return False
    return predicted.get("amount") == expected.get("amount")

def score_example(predicted_dict, expected_dict):
    scores = {}

    # product_name
    scores["product_name"] = (
        normalize(predicted_dict.get("product_name")) ==
        normalize(expected_dict.get("product_name"))
    )

    # brand
    scores["brand"] = (
        normalize(predicted_dict.get("brand")) ==
        normalize(expected_dict.get("brand"))
    )

    # category
    scores["category"] = (
        normalize(predicted_dict.get("category")) ==
        normalize(expected_dict.get("category"))
    )

    # price
    scores["price"] = compare_price(
        predicted_dict.get("price"),
        expected_dict.get("price")
    )

    # condition
    scores["condition"] = (
        normalize(predicted_dict.get("condition")) ==
        normalize(expected_dict.get("condition"))
    )

    return scores

print("Scoring functions ready!")


In [ ]:
valid_json_count = 0
field_scores = {
    "product_name": 0,
    "brand": 0,
    "category": 0,
    "price": 0,
    "condition": 0
}
failed_parses = []

for i, result in enumerate(results):
    predicted_dict = parse_json_response(result["predicted"])
    expected_dict = result["expected"]

    # If expected is a string, parse it too
    if isinstance(expected_dict, str):
        expected_dict = json.loads(expected_dict)

    if predicted_dict is None:
        failed_parses.append(i)
        continue

    valid_json_count += 1
    scores = score_example(predicted_dict, expected_dict)

    for field, correct in scores.items():
        if correct:
            field_scores[field] += 1

print(f"Valid JSON: {valid_json_count}/80 ({valid_json_count/80*100:.1f}%)")
print(f"\nField-level accuracy (out of {valid_json_count} valid responses):")
print("-" * 45)
for field, count in field_scores.items():
    if valid_json_count > 0:
        pct = count / valid_json_count * 100
    else:
        pct = 0
    print(f"  {field:15s}: {count}/{valid_json_count} ({pct:.1f}%)")
print(f"\nFailed to parse: {len(failed_parses)} examples")
if failed_parses:
    print(f"Failed indices: {failed_parses}")


In [ ]:
benchmark_report = {
    "model": "Qwen2.5-1.5B-Instruct",
    "quantization": "4-bit NF4",
    "total_examples": 80,
    "valid_json": valid_json_count,
    "valid_json_rate": round(valid_json_count / 80 * 100, 1),
    "field_accuracy": {},
    "failed_indices": failed_parses
}

for field, count in field_scores.items():
    if valid_json_count > 0:
        pct = round(count / valid_json_count * 100, 1)
    else:
        pct = 0
    benchmark_report["field_accuracy"][field] = {
        "correct": count,
        "total": valid_json_count,
        "percentage": pct
    }

# Save report
with open("benchmark_report.json", "w") as f:
    json.dump(benchmark_report, f, indent=2)

# Save all predictions for later analysis
with open("benchmark_predictions.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved: benchmark_report.json")
print("Saved: benchmark_predictions.json")
print("\n" + json.dumps(benchmark_report, indent=2))


In [ ]:
from google.colab import files

files.download("benchmark_report.json")
files.download("benchmark_predictions.json")
